# AOI-PCB: Model Training

This notebook walks through the full training pipeline for the IC keypoint detection model described in the paper.

The model learns to predict the four corner keypoints of an IC component placed on a PCB, trained entirely on **synthetic data** generated by `scripts/generate_dataset.py`.

### Prerequisites
1. Install the package: `pip install -e ".[dev]"`
2. Generate the dataset: `python scripts/generate_dataset.py`

### What this notebook covers
1. Data loading and encoding
2. Model architecture
3. Training with early stopping and LR scheduling
4. Loss and metric visualisation

## 1. Setup

In [ ]:
import shutil
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import tensorflow as tf
from keras import callbacks

from aoi_pcb.config_loader import Config
from aoi_pcb.data.encoder import DataEncoder
from aoi_pcb.model.architecture import build_model
from aoi_pcb.model.loss import custom_loss
from aoi_pcb.model.metric import KeypointAlignmentMetric

config = Config("../config.json")

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {len(gpus)}")
for gpu in gpus:
    print(f"  {gpu}")

## 2. Data Loading

The `DataEncoder` reads the generated PNG images and CSV label file, normalises pixel values to `[0, 1]`, and returns:
- `X` — `(N, 256, 256, 3)` image array
- `y` — `(N, 8)` label array (4 corners × x, y)
- `ref_coords` — reference corner positions (zero rotation, centred IC)
- `ref_center` — reference IC centre position

The reference keypoints are recorded as the first row of the CSV during generation and are used to initialise the alignment metric.

In [ ]:
encoder = DataEncoder(config)

data_dir = str(Path("..") / config.generator.train_data.data_dir)
label_path = Path("..") / config.generator.train_data.labels_dir / config.generator.train_data.label_file

X, y, ref_coords, ref_center = encoder(data_dir, label_path)

print(f"Images : {X.shape}  dtype={X.dtype}")
print(f"Labels : {y.shape}  dtype={y.dtype}")
print(f"ref_coords : {ref_coords}")
print(f"ref_center : {ref_center}")

## 3. Model Architecture

The model uses **MobileNetV2** (pre-trained on ImageNet) as a frozen feature extractor, followed by a lightweight regression head:

```
Input (256×256×3)
  └─ GaussianNoise(0.1)          # robustness to minor image variation
  └─ MobileNetV2 (frozen)        # ImageNet feature extraction
  └─ Dropout(0.3)
  └─ SeparableConv2D(8, 5×5)     # lightweight spatial aggregation
  └─ Flatten
  └─ Dense(512, ReLU)
  └─ Dropout(0.1)
  └─ Dense(8)                    # 4 corners × (x, y)
```

MobileNetV2 was chosen for its low parameter count, making the model suitable for deployment on edge hardware without cloud connectivity.

In [ ]:
input_shape = X.shape[1:]
output_shape = y.shape[1]

metric = KeypointAlignmentMetric(ref_center, ref_coords, config)
model = build_model(input_shape, output_shape)
model.summary()

## 4. Training

The model is compiled with:
- **Optimiser**: Adam
- **Loss**: Custom MSE + perpendicularity penalty (see `src/aoi_pcb/model/loss.py`)
- **Metric**: Weighted centre offset + angle error (see `src/aoi_pcb/model/metric.py`)

Two callbacks are used:
- `EarlyStopping` — halts training if validation loss stops improving
- `ReduceLROnPlateau` — reduces the learning rate when progress plateaus

All hyperparameters are read from `config.json`.

In [ ]:
run_dir = Path("../experiments") / f"notebook_run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
run_dir.mkdir(parents=True, exist_ok=True)
shutil.copy("../config.json", run_dir / "config.json")
model_path = run_dir / "model.keras"

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=config.training.optimizer_lr),
    loss=custom_loss,
    metrics=[metric.__call__],
)

es_args = config.get_init_kwargs("training.early_stopping")
lr_args = config.get_init_kwargs("training.lr_schedule")

training_callbacks = [
    callbacks.EarlyStopping(**es_args),
    callbacks.ReduceLROnPlateau(**lr_args),
    callbacks.CSVLogger(run_dir / f"training_{model_path.stem}.csv"),
]

history = model.fit(
    X,
    y,
    validation_split=config.training.val_split,
    epochs=config.training.n_epochs,
    callbacks=training_callbacks,
)

## 5. Save Model

In [ ]:
model.save(model_path)
print(f"Model saved to: {model_path}")

## 6. Results

Training and validation loss should converge by roughly epoch 600 (per the paper). The custom metric reflects the weighted error in IC centre position and rotation angle — lower is better.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history["loss"], label="Train", color="steelblue")
ax1.plot(history.history["val_loss"], label="Validation", color="tomato")
ax1.set_title("Loss per Epoch")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True)

metric_key = [k for k in history.history if "__call__" in k and "val" not in k][0]
val_metric_key = "val_" + metric_key

ax2.plot(history.history[metric_key], label="Train", color="steelblue")
ax2.plot(history.history[val_metric_key], label="Validation", color="tomato")
ax2.set_title("Alignment Metric per Epoch")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Metric")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f"Trained for {len(history.history['loss'])} epochs.")
print(f"Final train loss : {history.history['loss'][-1]:.6f}")
print(f"Final val   loss : {history.history['val_loss'][-1]:.6f}")